# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rohith84/Flyrank-Week-1-/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My Rule and Reason Codes

### Baseline Rule

The baseline prioritizes content pages that have high search visibility but low click-through rate (CTR).

A page receives a higher score when:
- Its impressions are high compared with other pages.
- Its CTR is low compared with other pages.

The score is intentionally simple and transparent so that it can be compared with a more advanced approach later.

### Reason Codes

- `HIGH_VISIBILITY_LOW_CTR` — high impressions and low CTR.
- `HIGH_VISIBILITY` — high impressions but CTR is not low.
- `LOW_CTR` — low CTR but impressions are not high.
- `MONITOR` — does not meet either condition.

### Actions

- `REVIEW_CTR` — review title and meta description.
- `PRIORITIZE_REVIEW` — prioritize the page for further content review.
- `MONITOR` — continue monitoring the page.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
from google.colab import userdata
from huggingface_hub import login
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN is not available in Colab Secrets.")

login(HF_TOKEN)

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("Hugging Face authentication configured.")

Hugging Face authentication configured.


In [6]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token found:", HF_TOKEN is not None)

login(HF_TOKEN)

Token found: True


In [7]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("DuckDB Hugging Face secret created.")

DuckDB Hugging Face secret created.


In [10]:
from google.colab import userdata
from huggingface_hub import whoami

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token exists:", HF_TOKEN is not None)

if HF_TOKEN:
    print("HF account:", whoami(token=HF_TOKEN)["name"])

Token exists: True
HF account: Rohith84


In [11]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
import duckdb
import os

HF_TOKEN = userdata.get("HF_TOKEN")

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance_sample.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print("Downloaded file:", file_path)
print("File exists:", os.path.exists(file_path))

fact_content_daily_performance_sample.pa(…): reconstructing file:   0%|          |  0.00B /  145MB            

fact_content_daily_performance_sample.pa(…): downloading bytes:           |  0.00B            

Downloaded file: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance_sample.parquet
File exists: True


In [12]:
con = duckdb.connect()

sample = con.execute(f"""
    SELECT *
    FROM read_parquet('{file_path}')
    LIMIT 50000
""").df()

print("Rows loaded:", len(sample))
print("Columns:", len(sample.columns))

sample.head()

Rows loaded: 50000
Columns: 31


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-01,client_3ffa76342f366962,content_1a6296faee432dae,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
1,2026-06-01,client_3ffa76342f366962,content_73f21e612565035a,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
2,2026-06-01,client_3ffa76342f366962,content_5a5be514ff559598,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
3,2026-06-01,client_3ffa76342f366962,content_05b377d0c8a5cfd8,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
4,2026-06-01,client_3ffa76342f366962,content_dc34c661d63e55a9,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06


In [13]:
sample["ctr"] = (
    sample["gsc_clicks"] /
    sample["gsc_impressions"].replace(0, np.nan)
).fillna(0)

In [14]:
high_impressions = sample["gsc_impressions"].quantile(0.75)
low_ctr = sample["ctr"].quantile(0.25)

print("High-impression threshold:", high_impressions)
print("Low-CTR threshold:", low_ctr)

High-impression threshold: 0.0
Low-CTR threshold: 0.0


In [15]:
sample["baseline_score"] = (
    (sample["gsc_impressions"] >= high_impressions).astype(int)
    +
    (sample["ctr"] <= low_ctr).astype(int)
)

In [16]:
sample["reason_code"] = np.select(
    [
        (sample["gsc_impressions"] >= high_impressions) &
        (sample["ctr"] <= low_ctr),

        (sample["gsc_impressions"] >= high_impressions),

        (sample["ctr"] <= low_ctr)
    ],
    [
        "HIGH_VISIBILITY_LOW_CTR",
        "HIGH_VISIBILITY",
        "LOW_CTR"
    ],
    default="MONITOR"
)

In [17]:
sample["action"] = np.select(
    [
        sample["reason_code"] == "HIGH_VISIBILITY_LOW_CTR",
        sample["reason_code"] == "HIGH_VISIBILITY",
        sample["reason_code"] == "LOW_CTR"
    ],
    [
        "REVIEW_CTR",
        "PRIORITIZE_REVIEW",
        "REVIEW_CTR"
    ],
    default="MONITOR"
)

In [18]:
baseline_queue = sample.sort_values(
    by=["baseline_score", "gsc_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

baseline_queue.head(20)

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month,ctr,baseline_score,reason_code,action
0,2026-06-01,client_62f4a7e64f5e0096,content_88ff1c6680db0a45,True,False,True,<NA>,4142,0,41907,...,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR
1,2026-06-01,client_62f4a7e64f5e0096,content_39584991d1c2b7a0,True,False,True,<NA>,1794,0,10042,...,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR
2,2026-06-01,client_62f4a7e64f5e0096,content_a603f13549019b16,True,False,True,<NA>,1544,0,11502,...,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR
3,2026-06-01,client_62f4a7e64f5e0096,content_d1582b1c3ba7f221,True,False,True,<NA>,1445,0,9992,...,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR
4,2026-06-01,client_62f4a7e64f5e0096,content_f4ce481bbfd43271,True,False,True,<NA>,1428,0,8550,...,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR
5,2026-06-01,client_62f4a7e64f5e0096,content_393cc2f021483a98,True,False,True,<NA>,1385,0,10990,...,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR
6,2026-06-01,client_62f4a7e64f5e0096,content_99d1bfa046d715ee,True,False,True,<NA>,1381,0,12342,...,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR
7,2026-06-01,client_62f4a7e64f5e0096,content_be5f11421172ad41,True,False,True,<NA>,1324,0,6766,...,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR
8,2026-06-01,client_62f4a7e64f5e0096,content_fb6a89e756e3556d,True,False,True,<NA>,1225,0,8348,...,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR
9,2026-06-01,client_62f4a7e64f5e0096,content_80057bf74597057f,True,False,True,<NA>,1160,0,8351,...,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR


In [19]:
os.makedirs("work/outputs", exist_ok=True)

baseline_queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Saved:", "work/outputs/baseline_action_score.csv")
print("Rows:", len(baseline_queue))

Saved: work/outputs/baseline_action_score.csv
Rows: 50000


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [20]:
top20 = baseline_queue.head(20).copy()

top20[[
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "baseline_score",
    "reason_code",
    "action"
]]

,content_hash_id,gsc_impressions,gsc_clicks,ctr,baseline_score,reason_code,action
0,content_88ff1c6680db0a45,4142,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR
1,content_39584991d1c2b7a0,1794,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR
2,content_a603f13549019b16,1544,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR
3,content_d1582b1c3ba7f221,1445,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR
4,content_f4ce481bbfd43271,1428,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR
5,content_393cc2f021483a98,1385,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR
6,content_99d1bfa046d715ee,1381,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR
7,content_be5f11421172ad41,1324,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR
8,content_fb6a89e756e3556d,1225,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR
9,content_80057bf74597057f,1160,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR


In [21]:
top20_review = top20[[
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "baseline_score",
    "reason_code",
    "action"
]].copy()

top20_review["confidence_note"] = np.where(
    top20_review["baseline_score"] == 2,
    "Strong baseline match: both high visibility and low CTR are present.",
    "Moderate baseline signal: only one primary condition is present."
)

top20_review["what_would_make_it_wrong"] = (
    "Search intent, SERP features, seasonality, or other page factors "
    "may explain the observed CTR."
)

top20_review

,content_hash_id,gsc_impressions,gsc_clicks,ctr,baseline_score,reason_code,action,confidence_note,what_would_make_it_wrong
0,content_88ff1c6680db0a45,4142,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR,Strong baseline match: both high visibility an...,"Search intent, SERP features, seasonality, or ..."
1,content_39584991d1c2b7a0,1794,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR,Strong baseline match: both high visibility an...,"Search intent, SERP features, seasonality, or ..."
2,content_a603f13549019b16,1544,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR,Strong baseline match: both high visibility an...,"Search intent, SERP features, seasonality, or ..."
3,content_d1582b1c3ba7f221,1445,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR,Strong baseline match: both high visibility an...,"Search intent, SERP features, seasonality, or ..."
4,content_f4ce481bbfd43271,1428,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR,Strong baseline match: both high visibility an...,"Search intent, SERP features, seasonality, or ..."
5,content_393cc2f021483a98,1385,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR,Strong baseline match: both high visibility an...,"Search intent, SERP features, seasonality, or ..."
6,content_99d1bfa046d715ee,1381,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR,Strong baseline match: both high visibility an...,"Search intent, SERP features, seasonality, or ..."
7,content_be5f11421172ad41,1324,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR,Strong baseline match: both high visibility an...,"Search intent, SERP features, seasonality, or ..."
8,content_fb6a89e756e3556d,1225,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR,Strong baseline match: both high visibility an...,"Search intent, SERP features, seasonality, or ..."
9,content_80057bf74597057f,1160,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR,Strong baseline match: both high visibility an...,"Search intent, SERP features, seasonality, or ..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [22]:
weak_picks = baseline_queue[
    baseline_queue["baseline_score"] < 2
].head(10)

weak_picks[[
    "content_hash_id",
    "gsc_impressions",
    "ctr",
    "baseline_score",
    "reason_code",
    "action"
]]

,content_hash_id,gsc_impressions,ctr,baseline_score,reason_code,action
49072,content_f107e54b10b43725,7179,0.002089,1,HIGH_VISIBILITY,PRIORITIZE_REVIEW
49073,content_8693b3c882998a58,4210,0.001188,1,HIGH_VISIBILITY,PRIORITIZE_REVIEW
49074,content_c556c7369fb2fd06,2505,0.004391,1,HIGH_VISIBILITY,PRIORITIZE_REVIEW
49075,content_9111c7d2691be9ad,2489,0.000402,1,HIGH_VISIBILITY,PRIORITIZE_REVIEW
49076,content_03621e012733c047,2300,0.000435,1,HIGH_VISIBILITY,PRIORITIZE_REVIEW
49077,content_815eb5ec41d452d4,2164,0.001848,1,HIGH_VISIBILITY,PRIORITIZE_REVIEW
49078,content_94497a88160b33d1,1998,0.002002,1,HIGH_VISIBILITY,PRIORITIZE_REVIEW
49079,content_7ff032d3024d35e4,1908,0.000524,1,HIGH_VISIBILITY,PRIORITIZE_REVIEW
49080,content_d6c6ff9b7d60af4f,1848,0.001082,1,HIGH_VISIBILITY,PRIORITIZE_REVIEW
49081,content_76c287232b54857b,1833,0.005456,1,HIGH_VISIBILITY,PRIORITIZE_REVIEW


## Weak Picks

Some pages can receive a recommendation even though they do not have both strong visibility and low CTR. These are weaker picks because the baseline only uses two signals.

A page with high impressions does not necessarily need a CTR-focused change, and a page with low CTR may have a valid explanation based on search intent or SERP context.

These weaker picks show the limitation of a simple baseline and provide a useful comparison point for later modeling.

## Leakage Check

The baseline uses only current-page search metrics:

- GSC impressions
- GSC clicks
- CTR derived from these values

No future-window metrics were used.

No product flags, model predictions, or future labels were used to create the baseline score.

The baseline is therefore intended as a transparent decision-support ranking rather than a prediction of future performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.